# 2. Count ecDNA with ecCount

This notebook downloads the released ecCount weights and runs the model on the
12 image sets from notebook 1, on the CPU (a few minutes in total).

The steps are the ones used for the paper: the image is resized to
1224 × 1024 px, pixels outside the ROI are set to zero, the network gives a
probability map, and local maxima of the smoothed map (above 0.35) are the
detected ecDNA. Each detection is drawn as a small diamond at full resolution
(the *peaks* mask); the probability map thresholded at 0.5 gives the
*threshold* mask.

**Needs:** notebook 1 run first (it provides `tutorial_data/`).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ecdna_bench

# All files of the tutorials live here (next to the notebook by default).
DATA = Path(os.environ.get("ECDNA_TUTORIAL_DATA", "tutorial_data")).expanduser().resolve()
print("ecdna_bench :", getattr(ecdna_bench, "__version__", "?"), "from", Path(ecdna_bench.__file__).parent)
print("data folder :", DATA)

import cv2
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)   # hide notes about extra TIFF tags

def read_rgb(path):
    """RGB image as uint8 (H, W, 3), read the same way as the benchmark."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_gray(path):
    """Single-channel image (TIFF or PNG); colour images are reduced by their maximum."""
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    img = np.asarray(img)
    return img.max(axis=2) if img.ndim == 3 else img

def load_sample():
    """The table written by notebook 1."""
    table = DATA / "sample.csv"
    if not table.is_file():
        raise FileNotFoundError(f"{table} not found. Run notebook 1 first "
                                "(or set ECDNA_TUTORIAL_DATA to its data folder).")
    return pd.read_csv(table)

sample = load_sample()
for col in ("rgb", "roi", "gt"):
    missing = [p for p in sample[col] if not (DATA / p).is_file()]
    assert not missing, f"{len(missing)} {col} file(s) missing; run notebook 1 again"
print(len(sample), "image sets ready")

## The weights

The weights (`eccount_best.pt`) are an asset of the repository's release
v1.0.0. The cell downloads them once into `tutorial_data/` and checks the
SHA-256 checksum published with the release. If you already have the file, set
`ECCOUNT_WEIGHTS=/path/to/eccount_best.pt` before starting Jupyter.

In [ ]:
import hashlib, urllib.request

ECCOUNT_URL = "https://github.com/Brunk-Lab/ecdna-bench/releases/download/v1.0.0/eccount_best.pt"
ECCOUNT_SHA256 = "7f11c52ccc12d50178a839cbfe584681e72fc00c3efb7acf816f92e09e228e32"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

WEIGHTS = Path(os.environ.get("ECCOUNT_WEIGHTS", DATA / "eccount_best.pt")).expanduser()
if not WEIGHTS.is_file():
    print("downloading", ECCOUNT_URL)
    part = WEIGHTS.with_name(WEIGHTS.name + ".part")
    req = urllib.request.Request(ECCOUNT_URL, headers={"User-Agent": "ecdna-bench-tutorial/1.0"})
    with urllib.request.urlopen(req, timeout=300) as r, open(part, "wb") as fh:
        while block := r.read(1 << 20):
            fh.write(block)
    part.replace(WEIGHTS)
digest = sha256(WEIGHTS)
if digest != ECCOUNT_SHA256:
    raise ValueError(f"{WEIGHTS} is not the released eccount_best.pt (SHA-256 {digest[:12]}...). "
                     "Delete it and run this cell again.")
print("weights OK:", WEIGHTS, f"({WEIGHTS.stat().st_size / 1e6:.0f} MB)")

In [ ]:
import torch
from ecdna_bench.eccount.model import ModelConfig, build_model
from ecdna_bench.eccount.infer import InferConfig, infer_one, load_checkpoint
from ecdna_bench.eccount.postprocess import PostprocessConfig, peaks_to_mask

DEVICE = os.environ.get("ECCOUNT_DEVICE", "cpu")
model = build_model(ModelConfig())            # the published architecture
try:
    ckpt = load_checkpoint(str(WEIGHTS), model, device=DEVICE)
except Exception as exc:                        # older checkpoints need the full unpickler
    if "weights_only" not in str(exc):
        raise
    ckpt = torch.load(str(WEIGHTS), map_location=DEVICE, weights_only=False)   # checksum verified above
    model.load_state_dict(ckpt["model_state_dict"])
model.eval().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ecCount U-Net: {n_params:,} parameters; checkpoint from epoch {ckpt.get('best_epoch', '?')}")

PP = PostprocessConfig()                         # frozen paper values
CFG = InferConfig(postprocess=PP, threshold_mask_cutoff=0.5, peaks_disk_radius=PP.point_disk_radius)
INPUT_H, INPUT_W = 1024, 1224
print(PP)

## Run ecCount

`run_eccount` reproduces the benchmark's inference for one image. The predicted
count is the number of objects in the peaks mask, counted with the same rule as
the gold standard (8-connected components of at least 3 px).

In [ ]:
import time
from ecdna_bench.evaluation import objects_from_mask

def count_objects(mask):
    return len(objects_from_mask(mask, min_area=3, connectivity=8, attach_mask=False))

def run_eccount(rgb_path, roi_path):
    rgb = read_rgb(rgb_path)
    orig_h, orig_w = rgb.shape[:2]
    small = cv2.resize(rgb, (INPUT_W, INPUT_H), interpolation=cv2.INTER_AREA)
    roi = (cv2.resize(read_gray(roi_path).astype(np.uint8), (INPUT_W, INPUT_H),
                      interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)
    small = small * roi[:, :, None]
    x = torch.from_numpy(small.astype(np.float32).transpose(2, 0, 1) / 255.0).unsqueeze(0).to(DEVICE)
    res = infer_one(x, roi, model, CFG)
    sx, sy = orig_w / INPUT_W, orig_h / INPUT_H
    peaks = [(int(round(px * sx)), int(round(py * sy)), s) for px, py, s in res.peaks]
    peaks_mask = peaks_to_mask(peaks, shape=(orig_h, orig_w), disk_radius=CFG.peaks_disk_radius)
    thr_mask = cv2.resize(res.threshold_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return res.prob_map, peaks, peaks_mask, thr_mask

OUT = DATA / "my_eccount"
(OUT / "peaks").mkdir(parents=True, exist_ok=True)
(OUT / "threshold").mkdir(parents=True, exist_ok=True)
released = pd.read_csv(DATA / "predictions.csv").query("method == 'ecCount (peaks)'").set_index("uid")["path"]

rows, t0 = [], time.time()
for r in sample.itertuples():
    t = time.time()
    prob, peaks, peaks_mask, thr_mask = run_eccount(DATA / r.rgb, DATA / r.roi)
    cv2.imwrite(str(OUT / "peaks" / f"{r.uid}.png"), peaks_mask)
    cv2.imwrite(str(OUT / "threshold" / f"{r.uid}.png"), thr_mask)
    rows.append({"cell_line": r.cell_line, "slot": r.slot, "uid": r.uid,
                 "gold_standard": count_objects(read_gray(DATA / r.gt)),
                 "ecCount_here": count_objects(peaks_mask),
                 "ecCount_released": count_objects(read_gray(DATA / released[r.uid])),
                 "seconds": round(time.time() - t, 1)})
    print(f"  {r.uid}: {rows[-1]['ecCount_here']} ecDNA ({rows[-1]['seconds']} s)", flush=True)
counts = pd.DataFrame(rows)
counts.to_csv(OUT / "counts.csv", index=False)
print(f"12 images in {time.time() - t0:.0f} s; masks in {OUT}")
counts.drop(columns="seconds")

`ecCount_released` is the count from the deposited ecCount (peaks) predictions,
which were computed on a GPU. CPU and GPU arithmetic differ slightly, so a few
peaks near the 0.35 cut-off can appear or disappear; the counts should agree
closely but not always exactly.

In [ ]:
ex = sample.set_index("uid").loc["ncih2170_facs_fish_0723_low_her2_52"]
prob, peaks, peaks_mask, _ = run_eccount(DATA / ex["rgb"], DATA / ex["roi"])
img, gt_mask = read_rgb(DATA / ex["rgb"]), read_gray(DATA / ex["gt"])
x0, y0, x1, y1 = 1247, 1225, 1458, 1353
prob_full = cv2.resize(prob, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)
pk = np.array([(px, py) for px, py, _ in peaks if x0 <= px < x1 and y0 <= py < y1])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].imshow(img[y0:y1, x0:x1]); ax[0].set_title("zoom of the example image")
ax[1].imshow(prob_full[y0:y1, x0:x1], vmin=0, vmax=1, cmap="magma"); ax[1].set_title("ecCount probability map")
ax[2].imshow(img[y0:y1, x0:x1])
ax[2].contour(gt_mask[y0:y1, x0:x1] > 0, levels=[0.5], colors="lime", linewidths=1)
if len(pk):
    ax[2].scatter(pk[:, 0] - x0, pk[:, 1] - y0, s=18, facecolors="none", edgecolors="magenta")
ax[2].set_title("gold standard (green) and ecCount peaks (magenta)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
for cl, g in counts.groupby("cell_line"):
    ax.scatter(g["gold_standard"], g["ecCount_here"], label=cl, s=40)
lim = [0, max(counts["gold_standard"].max(), counts["ecCount_here"].max()) * 1.05]
ax.plot(lim, lim, "k--", lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("gold-standard count"); ax.set_ylabel("ecCount count (this run)"); ax.legend()
plt.tight_layout(); plt.show()

## Next

**Notebook 3** scores these predictions, and the deposited predictions of all
benchmarked methods, against the gold standard.

To run ecCount on your own images, see route D in `docs/TUTORIAL_EXTERNAL.md`.